# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [ ]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [ ]:
df = pd.read_csv("data/aviation_cleaned.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [ ]:
df["Aircraft.Size"] = np.where(
    df["Total.Passengers"] <= 20,
    "Small Aircraft",
    "Large Aircraft"
)

df["Aircraft.Size"].value_counts()

In [ ]:
df[[
    "Make",
    "Model",
    "Make.Model",
    "Aircraft.Size",
    "Total.Passengers",
    "Fatal.Serious.Injury.Rate",
    "Aircraft.Destroyed"
]].head()

#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

In [ ]:
make_summary = df.groupby(["Aircraft.Size", "Make"]).agg(
    accident_count=("Make", "count"),
    avg_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "mean"),
    median_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "median"),
    destruction_rate=("Aircraft.Destroyed", "mean")
).reset_index()

make_summary.head()

In [ ]:
robust_make_summary = make_summary[make_summary["accident_count"] >= 50].copy()

robust_make_summary.shape

**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
small_makes = (
    robust_make_summary[robust_make_summary["Aircraft.Size"] == "Small Aircraft"]
    .sort_values(
        ["avg_fatal_serious_rate", "destruction_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

small_makes

In [ ]:
small_top_10_makes = small_makes.head(10)["Make"]

small_make_data = df[
    (df["Aircraft.Size"] == "Small Aircraft") &
    (df["Make"].isin(small_top_10_makes))
].copy()

plt.figure(figsize=(14, 7))
sns.violinplot(
    data=small_make_data,
    x="Make",
    y="Fatal.Serious.Injury.Rate"
)
plt.title("Distribution of Fatal/Serious Injury Rates for Top 10 Small Aircraft Makes")
plt.xlabel("Make")
plt.ylabel("Fatal/Serious Injury Rate")
plt.xticks(rotation=75)
plt.show()

**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
large_makes = (
    robust_make_summary[robust_make_summary["Aircraft.Size"] == "Large Aircraft"]
    .sort_values(
        ["avg_fatal_serious_rate", "destruction_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

large_makes

In [ ]:
large_top_10_makes = large_makes.head(10)["Make"]

large_make_data = df[
    (df["Aircraft.Size"] == "Large Aircraft") &
    (df["Make"].isin(large_top_10_makes))
].copy()

plt.figure(figsize=(14, 7))
sns.stripplot(
    data=large_make_data,
    x="Make",
    y="Fatal.Serious.Injury.Rate",
    jitter=True,
    alpha=0.6
)
plt.title("Distribution of Fatal/Serious Injury Rates for Top 10 Large Aircraft Makes")
plt.xlabel("Make")
plt.ylabel("Fatal/Serious Injury Rate")
plt.xticks(rotation=75)
plt.show()

**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

In [ ]:
small_destruction = (
    robust_make_summary[robust_make_summary["Aircraft.Size"] == "Small Aircraft"]
    .sort_values(
        ["destruction_rate", "avg_fatal_serious_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

small_destruction

In [ ]:
large_destruction = (
    robust_make_summary[robust_make_summary["Aircraft.Size"] == "Large Aircraft"]
    .sort_values(
        ["destruction_rate", "avg_fatal_serious_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

large_destruction

In [ ]:
destruction_top_makes = pd.concat([small_destruction, large_destruction])

plt.figure(figsize=(14, 7))
sns.barplot(
    data=destruction_top_makes,
    x="Make",
    y="destruction_rate",
    hue="Aircraft.Size"
)
plt.title("Lowest Aircraft Destruction Rate by Make and Aircraft Size")
plt.xlabel("Make")
plt.ylabel("Destruction Rate")
plt.xticks(rotation=75)
plt.show()

#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

In [ ]:
### Make-Level Findings

The analysis separated aircraft into small and large aircraft using a 20-passenger threshold. Make-level comparisons were limited to makes with at least 50 accident records to avoid weak conclusions based on very small samples.

For small aircraft, the strongest recommendations are the makes with both low fatal/serious injury rates and low aircraft destruction rates. For large aircraft, the same logic was applied separately because large passenger aircraft operate under different conditions and carry more passengers.

The plots show that some makes have lower average severe injury rates, while others show wider variation across accidents. Makes with consistently low injury rates and low destruction rates are stronger recommendations for the insurer.

### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

In [ ]:
plane_type_summary = df.groupby("Aircraft.Size").agg(
    accident_count=("Aircraft.Size", "count"),
    avg_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "mean"),
    median_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "median"),
    destruction_rate=("Aircraft.Destroyed", "mean")
).reset_index()

plane_type_summary

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(
    data=plane_type_summary,
    x="Aircraft.Size",
    y="avg_fatal_serious_rate"
)
plt.title("Average Fatal/Serious Injury Rate by Aircraft Size")
plt.xlabel("Aircraft Size")
plt.ylabel("Average Fatal/Serious Injury Rate")
plt.show()

**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

In [ ]:
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df,
    x="Aircraft.Size",
    y="Fatal.Serious.Injury.Rate"
)
plt.title("Distribution of Fatal/Serious Injury Rates by Aircraft Size")
plt.xlabel("Aircraft Size")
plt.ylabel("Fatal/Serious Injury Rate")
plt.show()

### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

In [ ]:
model_summary = df.groupby(["Aircraft.Size", "Make", "Model", "Make.Model"]).agg(
    accident_count=("Make.Model", "count"),
    avg_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "mean"),
    median_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "median"),
    destruction_rate=("Aircraft.Destroyed", "mean")
).reset_index()

robust_model_summary = model_summary[model_summary["accident_count"] >= 10].copy()

robust_model_summary.head()

In [ ]:
large_plane_recommendations = (
    robust_model_summary[robust_model_summary["Aircraft.Size"] == "Large Aircraft"]
    .sort_values(
        ["avg_fatal_serious_rate", "destruction_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

large_plane_recommendations

In [ ]:
small_plane_recommendations = (
    robust_model_summary[robust_model_summary["Aircraft.Size"] == "Small Aircraft"]
    .sort_values(
        ["avg_fatal_serious_rate", "destruction_rate", "accident_count"],
        ascending=[True, True, False]
    )
    .head(15)
)

small_plane_recommendations

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(
    data=large_plane_recommendations,
    x="Make.Model",
    y="avg_fatal_serious_rate"
)
plt.title("Recommended Large Aircraft Models by Low Fatal/Serious Injury Rate")
plt.xlabel("Aircraft Model")
plt.ylabel("Average Fatal/Serious Injury Rate")
plt.xticks(rotation=85)
plt.show()

In [ ]:
plt.figure(figsize=(14, 7))
sns.barplot(
    data=small_plane_recommendations,
    x="Make.Model",
    y="avg_fatal_serious_rate"
)
plt.title("Recommended Small Aircraft Models by Low Fatal/Serious Injury Rate")
plt.xlabel("Aircraft Model")
plt.ylabel("Average Fatal/Serious Injury Rate")
plt.xticks(rotation=85)
plt.show()

### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.

In [ ]:
weather_summary = df.groupby("Weather.Condition").agg(
    accident_count=("Weather.Condition", "count"),
    avg_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "mean"),
    destruction_rate=("Aircraft.Destroyed", "mean")
).reset_index().sort_values("avg_fatal_serious_rate", ascending=False)

weather_summary

In [ ]:
phase_summary = df.groupby("Broad.phase.of.flight").agg(
    accident_count=("Broad.phase.of.flight", "count"),
    avg_fatal_serious_rate=("Fatal.Serious.Injury.Rate", "mean"),
    destruction_rate=("Aircraft.Destroyed", "mean")
).reset_index()

phase_summary = phase_summary[phase_summary["accident_count"] >= 50]

phase_summary = phase_summary.sort_values("avg_fatal_serious_rate", ascending=False)

phase_summary

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=phase_summary,
    x="Broad.phase.of.flight",
    y="avg_fatal_serious_rate"
)
plt.title("Average Fatal/Serious Injury Rate by Phase of Flight")
plt.xlabel("Phase of Flight")
plt.ylabel("Average Fatal/Serious Injury Rate")
plt.xticks(rotation=75)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=weather_summary,
    x="Weather.Condition",
    y="avg_fatal_serious_rate"
)
plt.title("Average Fatal/Serious Injury Rate by Weather Condition")
plt.xlabel("Weather Condition")
plt.ylabel("Average Fatal/Serious Injury Rate")
plt.xticks(rotation=45)
plt.show()

### Weather Condition Discussion

Weather condition was analyzed as one factor that may affect accident severity. The table compares accident count, average fatal/serious injury rate, and aircraft destruction rate by weather condition.

Weather categories with higher fatal/serious injury rates may indicate more severe accident outcomes. This supports the idea that operating environment can influence passenger injury risk.

### Phase of Flight Discussion

Phase of flight was analyzed as a second factor affecting accident outcomes. Only phases with at least 50 records were included to avoid weak comparisons.

The results suggest that accident severity varies by flight phase. Some phases may show higher fatal/serious injury rates because they involve higher speeds, lower altitude, or less time available for recovery.

## Final Summary and Recommendations

This analysis used cleaned aviation accident data from 1983 onward to align with the client's requirement that aircraft makes and models could still reasonably be active. Aircraft were separated into small and large categories using a 20-passenger threshold.

To make findings more statistically reliable, make-level recommendations used a minimum threshold of 50 accident records, while model-level recommendations used a minimum threshold of 10 accident records.

The analysis focused on two key safety metrics: fatal/serious injury rate and aircraft destruction rate. Recommended makes and models were selected based on lower fatal/serious injury rates, lower destruction rates, and sufficient sample size.

The analysis also found that weather condition and phase of flight are meaningful factors in accident outcomes. Weather condition showed differences in injury severity, and phase of flight showed differences in both injury severity and destruction outcomes.

Overall, the safest recommendations are aircraft makes and models that consistently show low severe injury rates and low destruction rates across a sufficient number of accident records.